# 01. 2024·2025 교통카드 데이터 EDA

두 기준일의 데이터 구조, 노선 수, 결측치, 이용량을 확인한다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 노트북을 Analysis 폴더 또는 저장소 루트에서 실행해도 동작하도록 설정
ROOT = Path.cwd()
if not (ROOT / 'gtx_a_seoul_bus_outputs').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'gtx_a_seoul_bus_outputs' / 'transport_card'
FILE_2024 = DATA_DIR / 'gtx_a_transport_card_20241017_raw_with_coords_api_filled.csv'
FILE_2025 = DATA_DIR / 'gtx_a_transport_card_20251016_raw_with_coords.csv'

# Windows 한글 폰트 우선 적용
font_candidates = ['Malgun Gothic', 'NanumGothic', 'AppleGothic']
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
for font_name in font_candidates:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

df24 = pd.read_csv(FILE_2024, encoding='utf-8-sig')
df25 = pd.read_csv(FILE_2025, encoding='utf-8-sig')
for df in (df24, df25):
    df['utztn_nope'] = pd.to_numeric(df['utztn_nope'], errors='coerce')
    df['utztn_dstnc'] = pd.to_numeric(df['utztn_dstnc'], errors='coerce')
print(f'2024: {df24.shape}, 2025: {df25.shape}')

## 기본 구조와 노선별 이용량

In [ ]:
route_summary = pd.DataFrame({
    '2024 행 수': df24.groupby('query_route_no').size(),
    '2025 행 수': df25.groupby('query_route_no').size(),
    '2024 이용인원': df24.groupby('query_route_no')['utztn_nope'].sum(),
    '2025 이용인원': df25.groupby('query_route_no')['utztn_nope'].sum(),
}).fillna(0).sort_index()
route_summary['증감률(%)'] = np.where(
    route_summary['2024 이용인원'] != 0,
    (route_summary['2025 이용인원'] - route_summary['2024 이용인원']) / route_summary['2024 이용인원'] * 100,
    np.nan,
)
display(route_summary)

In [ ]:
missing = pd.DataFrame({
2024: df24.isna().sum(),
2025: df25.isna().sum(),
})
display(missing[missing.sum(axis=1) > 0].sort_values(2024, ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, df, year in zip(axes, [df24, df25], ['2024', '2025']):
    values = df.groupby('query_route_no')['utztn_nope'].sum().sort_values(ascending=False)
    values.plot.bar(ax=ax, color='#4472C4')
    ax.set_title(f'{year} 노선별 이용인원')
    ax.set_xlabel('노선번호')
    ax.set_ylabel('이용인원')
    ax.tick_params(axis='x', rotation=70)
plt.tight_layout()
plt.show()

## 2024·2025 노선 구성 비교

In [ ]:
routes24 = set(df24['query_route_no'].dropna().astype(str))
routes25 = set(df25['query_route_no'].dropna().astype(str))
route_comparison = pd.DataFrame({
    '2024·2025 공통': pd.Series(sorted(routes24 & routes25, key=str)),
    '2024년에만 존재': pd.Series(sorted(routes24 - routes25, key=str)),
    '2025년에만 존재': pd.Series(sorted(routes25 - routes24, key=str)),
})
display(route_comparison)
print(f'2024 노선 수: {len(routes24)} / 2025 노선 수: {len(routes25)}')

demand = pd.DataFrame({
    '2024': df24.groupby('query_route_no')['utztn_nope'].sum(),
    '2025': df25.groupby('query_route_no')['utztn_nope'].sum(),
}).fillna(0)
demand['증감'] = demand['2025'] - demand['2024']
demand['증감률(%)'] = np.where(demand['2024'] != 0, demand['증감'] / demand['2024'] * 100, np.nan)
display(demand.sort_values('증감', ascending=False))

ax = demand[['2024', '2025']].sort_index().plot.bar(figsize=(17, 6), color=['#A5A5A5', '#4472C4'])
ax.set_title('노선별 2024·2025 이용인원 비교')
ax.set_xlabel('노선번호')
ax.set_ylabel('이용인원')
ax.tick_params(axis='x', rotation=70)
ax.legend(title='기준연도')
plt.tight_layout()
plt.show()

## 정류장별 수요 시각화

In [ ]:
def stop_summary(df, id_col, name_col, lat_col, lon_col, n=15):
    return (df.groupby([id_col, name_col], dropna=False)
              .agg(이용인원=('utztn_nope', 'sum'), 위도=(lat_col, 'first'), 경도=(lon_col, 'first'), 이용건수=('utztn_nope', 'size'))
              .sort_values('이용인원', ascending=False).head(n))

for year, df in [('2024', df24), ('2025', df25)]:
    print(f'===== {year} 승차 상위 정류장 =====')
    display(stop_summary(df, 'ride_sttn_id', '승차정류장명', '승차위도', '승차경도'))
    print(f'===== {year} 하차 상위 정류장 =====')
    display(stop_summary(df, 'goff_sttn_id', '하차정류장명', '하차위도', '하차경도'))

def plot_top_stops(df, year, id_col, name_col, title, color):
    s = df.groupby([id_col, name_col])['utztn_nope'].sum().sort_values(ascending=False).head(15).sort_values()
    labels = [f'{idx[1]}\n({idx[0]})' for idx in s.index]
    ax = s.set_axis(labels).plot.barh(figsize=(10, 7), color=color)
    ax.set_title(f'{year} {title} 상위 15개')
    ax.set_xlabel('이용인원')
    ax.set_ylabel('정류장')
    plt.tight_layout()
    plt.show()

plot_top_stops(df24, '2024', 'ride_sttn_id', '승차정류장명', '승차 정류장 이용량', '#70AD47')
plot_top_stops(df25, '2025', 'ride_sttn_id', '승차정류장명', '승차 정류장 이용량', '#4472C4')
plot_top_stops(df24, '2024', 'goff_sttn_id', '하차정류장명', '하차 정류장 이용량', '#ED7D31')
plot_top_stops(df25, '2025', 'goff_sttn_id', '하차정류장명', '하차 정류장 이용량', '#A5A5A5')

## 노선별 시간대 이용 비중

In [ ]:
def hourly_route_share(df):
    work = df.copy()
    work['승차시간'] = pd.to_datetime(work['ride_dt'].astype(str), format='%Y%m%d%H%M%S', errors='coerce').dt.hour
    work = work.dropna(subset=['승차시간', 'query_route_no'])
    work['승차시간'] = work['승차시간'].astype(int)
    hourly = (work.groupby(['query_route_no', '승차시간'])['utztn_nope']
                   .sum().rename('이용인원').reset_index())
    hourly['노선별 비중(%)'] = (hourly['이용인원'] /
        hourly.groupby('query_route_no')['이용인원'].transform('sum') * 100)
    count_table = hourly.pivot(index='query_route_no', columns='승차시간', values='이용인원').fillna(0)
    share_table = hourly.pivot(index='query_route_no', columns='승차시간', values='노선별 비중(%)').fillna(0)
    full_hours = list(range(24))
    count_table = count_table.reindex(columns=full_hours, fill_value=0)
    share_table = share_table.reindex(columns=full_hours, fill_value=0)
    return hourly, count_table, share_table

hourly24, hourly_count24, hourly_share24 = hourly_route_share(df24)
hourly25, hourly_count25, hourly_share25 = hourly_route_share(df25)

# 각 노선×시간대의 실제 이용인원
display(hourly_count24.astype(int).rename_axis('2024 노선'))
display(hourly_count25.astype(int).rename_axis('2025 노선'))

# 각 노선의 전체 이용인원을 100으로 보았을 때 시간대별 비중(%)
display(hourly_share24.round(2).rename_axis('2024 노선'))
display(hourly_share25.round(2).rename_axis('2025 노선'))

In [ ]:
def plot_hourly_share(share_table, year):
    fig, ax = plt.subplots(figsize=(18, max(7, len(share_table) * 0.45)))
    image = ax.imshow(share_table.values, aspect='auto', cmap='YlOrRd')
    ax.set_title(f'{year} 노선별 시간대 이용 비중(%)')
    ax.set_xlabel('승차 시간대')
    ax.set_ylabel('노선번호')
    ax.set_xticks(range(24))
    ax.set_xticklabels([f'{h}시' for h in range(24)])
    ax.set_yticks(range(len(share_table.index)))
    ax.set_yticklabels(share_table.index)
    fig.colorbar(image, ax=ax, label='노선 내 비중(%)')
    plt.tight_layout()
    plt.show()

plot_hourly_share(hourly_share24, '2024')
plot_hourly_share(hourly_share25, '2025')

시간대별 비중은 각 노선의 전체 이용인원을 100%로 놓고 계산한다. 예를 들어 특정 노선의 08시 값이 15라면, 해당 노선 전체 이용인원의 15%가 08시에 발생했다는 의미이다.